#### Packages & Clean Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Spectral data
# s_IMPROV = pd.read_csv("C:/Users/monik/Documents/Paris/Internship/Analyses/Pilot-Data/Keith-Data-1/Big-Sets/Clean/spectral_data_Clean_ALL_IMPROV_pilots.csv",
#                  index_col=0)
s1 = pd.read_csv("C:/Users/monik/Documents/Paris/Internship/Analyses/Pilot-Data/Keith-2-Data/spectral_data_999_IMPROV.csv",
                 index_col=0)

participant_ID = 999
condition = 'IMPROV'

In [3]:
# Replace blanks with np.nan
s1["MIDI_NOTE"] = s1["MIDI_NOTE"].replace([],np.nan)
s1["MIDI_NOTE"] = s1["MIDI_NOTE"].replace('',np.nan)

In [5]:
# Replace 'note' with 1 and 'glide' with 0.
s1['NG_type'] = s1['NG_type'].replace(['note'], 1)
s1['NG_type'] = s1['NG_type'].replace(['glide'], 0)

# Initialize arrays
nr_notes = np.zeros(30)
note_durations = []; 
add_duration = note_durations.append
mean_midi_notes = []
add_mean_note = mean_midi_notes.append

# Loop through all songs: 30 songs for improv, 16 for song forms !!!!
n_songs = s1.SONG_ID.unique().shape[0]

for i in range(1,n_songs+1):
    
    # Select NG type column for each song and replace nan with 0.
    song1 = s1[s1['SONG_ID'] == i]['NG_type']
    song1 = song1.fillna(value=0)
    song1 = song1.to_numpy()
    
    # Detect note change
    change = np.diff(song1)
    
    # Get index/time of start and end note change.
    start_idx = np.where(change == 1)[0] 
    end_idx   = np.where(change == -1)[0]
    start_times = s1[s1['SONG_ID'] == 1]["TIME_FRAME"].to_numpy()[start_idx]
    end_times = s1[s1['SONG_ID'] == 1]["TIME_FRAME"].to_numpy()[end_idx]
    
    # Get note duration and append to list
    dur = end_times - start_times
    add_duration(dur)
    
    
    # Get number of notes & append to list
    note_nr = start_idx.shape[0]
    nr_notes[i-1] = note_nr
    
    # For each note interval, get the mean of all midi notes.
    for j in range(note_nr):
            mean_note = np.nanmean(a = s1[s1['SONG_ID'] == i]["MIDI_NOTE"].to_numpy()[start_idx[j]:end_idx[j]])        
            add_mean_note(mean_note)

# Convert to numpy array
note_durations = list(np.concatenate(note_durations))

In [6]:
# Create a dataframe of MIDI NOTEs and their durations for all songs.
data_Weiss = pd.DataFrame()
data_Weiss['NOTE'] = mean_midi_notes
data_Weiss['DURATION'] = note_durations

# Save the dataframe to csv.
data_Weiss.to_csv(f"Weiss_data_{participant_ID}_{condition}.csv")